In [9]:
# CELL 2: Visual Feature Extraction with CLIP
import os
import cv2
import torch
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm
from transformers import CLIPProcessor, CLIPModel

PROJECT_PATH = "Thesis_Data"

# 1. Load CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading CLIP model on {device}...")
model_clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor_clip = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def extract_clip_features(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0: fps = 30
    
    frame_features = []
    count = 0
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        # Extract 1 frame per second
        if count % int(fps) == 0:
            # Convert BGR (OpenCV) to RGB (PIL)
            img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(img)
            
            inputs = processor_clip(images=pil_img, return_tensors="pt").to(device)
            
            with torch.no_grad():
                # --- THE BULLETPROOF FIX ---
                # 1. Run the vision part of the model explicitly (This outputs the "Object")
                vision_outputs = model_clip.vision_model(pixel_values=inputs['pixel_values'])
                
                # 2. Extract the raw tensor from inside that Object
                pooled_tensor = vision_outputs.pooler_output 
                
                # 3. Project it into the final 512-dimensional CLIP space
                features = model_clip.visual_projection(pooled_tensor)
                
            frame_features.append(features.cpu().numpy().flatten())
        count += 1
        
    cap.release()
    
    if not frame_features:
        return np.zeros(512) # CLIP patch32 outputs 512 dimensions
        
    # Average the features across the whole video
    return np.mean(frame_features, axis=0)

# 2. Process all videos
visual_features_dict_clip2 = {}
video_files = [f for f in os.listdir(f"{PROJECT_PATH}/videos2") if f.endswith('.mp4')]

print("Extracting CLIP features (Visual)...")
for v_file in tqdm(video_files):
    v_id = v_file.replace(".mp4", "")
    v_path = f"{PROJECT_PATH}/videos2/{v_file}"
    try:
        visual_features_dict_clip2[v_id] = extract_clip_features(v_path)
    except Exception as e:
        print(f"Error on {v_id}: {e}")

# 3. Save with a NEW name
np.save(f"{PROJECT_PATH}/visual_features_clip2.npy", visual_features_dict_clip2)
print("CLIP Visual features saved successfully!")

Loading CLIP model on cpu...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting CLIP features (Visual)...


  0%|          | 0/141 [00:00<?, ?it/s]

CLIP Visual features saved successfully!


In [10]:
# CELL 3: Audio Feature Extraction with VGGishm

# 1. Load VGGish from PyTorch Hub
print("Loading VGGish model...")
# We use harritaylor's implementation which is the standard PyTorch port for VGGish
vggish = torch.hub.load('harritaylor/torchvggish', 'vggish')
vggish.eval()

def extract_vggish_features(audio_path):
    # The harritaylor VGGish port takes a wav file path directly,
    # automatically resamples it to 16kHz, computes the mel-spectrogram, 
    # and runs it through the network!
    with torch.no_grad():
        # Outputs shape: [number_of_seconds, 128]
        embeddings = vggish.forward(audio_path)
    
    # Average the embeddings over time to get one 128-D vector for the whole ad
    # Convert tensor to numpy
    features_np = embeddings.cpu().numpy()
    
    if len(features_np) == 0:
        return np.zeros(128)
        
    return np.mean(features_np, axis=0)

# 2. Process all audio files
audio_features_dict_vggish2 = {}
audio_files = [f for f in os.listdir(f"{PROJECT_PATH}/audio2") if f.endswith('.wav')]

print("Extracting VGGish features (Audio2)...")
for a_file in tqdm(audio_files):
    v_id = a_file.replace(".wav", "")
    a_path = f"{PROJECT_PATH}/audio2/{a_file}"
    try:
        audio_features_dict_vggish2[v_id] = extract_vggish_features(a_path)
    except Exception as e:
        print(f"Error on {v_id}: {e}")

# 3. Save with a NEW name
np.save(f"{PROJECT_PATH}/audio_features_vggish2.npy", audio_features_dict_vggish2)
print("VGGish Audio features saved successfully!")

Loading VGGish model...


Using cache found in C:\Users\yasam/.cache\torch\hub\harritaylor_torchvggish_master


Extracting VGGish features (Audio2)...


c:\Users\yasam\Desktop\thesis-notebooks\venv\Lib\site-packages\torch\serialization.py:1832: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  result = unpickler.load()


  0%|          | 0/141 [00:00<?, ?it/s]

VGGish Audio features saved successfully!
